# Oanda Tradable Instruments

Fetches every instrument available on your Oanda **practice** account and renders it as a styled table.

- **API:** Oanda v20 REST via `oandapyV20`
- **Auth:** reads `OANDA_API_TOKEN` and `OANDA_ACCOUNT_ID` from `.env`
- **Re-run Cell 4 onward** to refresh the data at any time.

In [6]:
from __future__ import annotations

import os
from datetime import datetime, timezone

import pandas as pd
from dotenv import load_dotenv
import oandapyV20
import oandapyV20.endpoints.accounts as accounts
from IPython.display import display, HTML

In [7]:
load_dotenv()

API_TOKEN  = os.environ["OANDA_API_TOKEN"]
ACCOUNT_ID = os.environ["OANDA_ACCOUNT_ID"]

client = oandapyV20.API(access_token=API_TOKEN, environment="practice")
print(f"Client ready  —  account: {ACCOUNT_ID}")

Client ready  —  account: 101-004-20758957-001


In [8]:
r    = accounts.AccountInstruments(accountID=ACCOUNT_ID)
resp = client.request(r)
raw  = resp["instruments"]

print(f"Fetched {len(raw)} instruments from Oanda.")

Fetched 123 instruments from Oanda.


In [9]:
FIELDS: dict[str, str] = {
    "name"             : "Symbol",
    "type"             : "Type",
    "displayName"      : "Name",
    "pipLocation"      : "Pip Location",
    "displayPrecision" : "Decimals",
    "minimumTradeSize" : "Min Size",
    "maximumOrderUnits": "Max Units",
    "marginRate"       : "Margin %",
}

df = pd.DataFrame(
    [{friendly: inst.get(raw_key) for raw_key, friendly in FIELDS.items()} for inst in raw]
)

df["Margin %"]  = pd.to_numeric(df["Margin %"])  * 100
df["Min Size"]  = pd.to_numeric(df["Min Size"])
df["Max Units"] = pd.to_numeric(df["Max Units"])

df = df.sort_values(["Type", "Symbol"]).reset_index(drop=True)
df.index = df.index + 1  # 1-based row numbers

df.head(3)

,Symbol,Type,Name,Pip Location,Decimals,Min Size,Max Units,Margin %
1,AU200_AUD,CFD,Australia 200,0,1,0.1,1250,5.0
2,BCO_USD,CFD,Brent Crude Oil,-2,3,1.0,100000,10.0
3,CH20_CHF,CFD,Switzerland 20,0,1,0.1,500,10.0


In [10]:
fetched_at = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")

df_display = df.copy()
df_display["Margin %"] = df_display["Margin %"].map("{:.2f}%".format)
df_display["Min Size"] = df_display["Min Size"].map("{:,.0f}".format)
df_display["Max Units"] = df_display["Max Units"].map("{:,.0f}".format)

table_html = df_display.to_html(classes="oanda-table", index=True)

html = f"""
<style>
  .oanda-table {{ border-collapse: collapse; font-size: 13px; font-family: sans-serif; background-color: #ffffff; }}
  .oanda-table th {{ background-color: #2c3e50; color: #ffffff; padding: 6px 14px; text-align: left; white-space: nowrap; }}
  .oanda-table td {{ padding: 5px 14px; border-bottom: 1px solid #ddd; color: #000000; background-color: #ffffff; }}
  .oanda-table tr:nth-child(even) td {{ background-color: #f4f6f8; color: #000000; }}
  .oanda-table tr:hover td {{ background-color: #d0e8fb; color: #000000; }}
</style>
<p style="font-size:1.1em;font-weight:bold;margin-bottom:8px;color:#000000;">
  Oanda Practice — Tradable Instruments &nbsp;&nbsp;<span style="font-weight:normal;font-size:0.9em;color:#555555;">({fetched_at})</span>
</p>
{table_html}
"""

display(HTML(html))

summary = df.groupby("Type").size().rename("Count")
print("\nInstrument count by type:")
print(summary.to_string())

,Symbol,Type,Name,Pip Location,Decimals,Min Size,Max Units,Margin %
1,AU200_AUD,CFD,Australia 200,0,1,0,"1,250",5.00%
2,BCO_USD,CFD,Brent Crude Oil,-2,3,1,"100,000",10.00%
3,CH20_CHF,CFD,Switzerland 20,0,1,0,500,10.00%
4,CHINAH_HKD,CFD,China H Shares,0,1,0,500,10.00%
5,CN50_USD,CFD,China A50,0,1,0,500,10.00%
6,CORN_USD,CFD,Corn,-2,3,1,"1,500,000",10.00%
7,DE10YB_EUR,CFD,Bund,-2,3,1,"150,000",20.00%
8,DE30_EUR,CFD,Germany 30,0,1,0,"1,000",5.00%
9,ESPIX_EUR,CFD,Spain 35,0,1,0,500,10.00%
10,EU50_EUR,CFD,Europe 50,0,1,0,"3,000",5.00%



Instrument count by type:
Type
CFD         34
CURRENCY    68
METAL       21
